# Categorize Responses in the Behavioural Set

In [2]:
import sys, os

PARENT_DIR = os.path.abspath(os.path.join(os.getcwd(),".."))
if PARENT_DIR not in sys.path:
    sys.path.insert(0,PARENT_DIR)

In [26]:
import torch
import json
from typing import List, Literal, Optional
from enum import Enum
import datetime
from dataclasses import dataclass
from pydantic import BaseModel, Field
from transformers import AutoTokenizer, AutoModelForTokenClassification
from config import settings
from gcp_utils import download_from_gcs
from inference.v01.inference_utils import predict_word_level, word_labels_to_spans

In [ ]:
@dataclass 
class ErrorTaxonomy: 
    type_0 : str = "Irrelevant Span Mislabeling"
    type_1: str = "Missing Expected Entity"
    type_2: str = "Tokenization Artifacts"
    type_3: str = "Incorrect Polarity Assignment"
    type_4: str = "BIO Sequencing Errors"
    type_5: str = "Boundary Overreach"
    type_6: str = "Boundary Undereach"
    type_7: str = "Model Overgeneralization"

# class ErrorTaxonomy: 
#     irrelevant_span_mislabeling : str = "Irrelevant Span Mislabeling"
#     missing_expected_entity: str = "Missing Expected Entity"
#     tokenization_artifacts: str = "Tokenization Artifacts"
#     incorrect_polarity_assignment: str = "Incorrect Polarity Assignment"
#     bio_sequencing_errors: str = "BIO Sequencing Errors"
#     boundary_overreach: str = "Boundary Overreach"
#     boundary_undereach: str = "Boundary Undereach"
#     model_overgeneralization: str = "Model Overgeneralization"

ERROR_TAXONOMY = {
    "0":ErrorTaxonomy.type_0,
    "1":ErrorTaxonomy.type_1,
    "2":ErrorTaxonomy.type_2,
    "3":ErrorTaxonomy.type_3,
    "4":ErrorTaxonomy.type_4,
    "5":ErrorTaxonomy.type_5,
    "6":ErrorTaxonomy.type_6,
    "7":ErrorTaxonomy.type_7
}

MODEL_NAME= "dmis-lab/biobert-base-cased-v1.1"
RUN_IDX ="2"# BEST RUN
VERSION = "v01"
# inference pipeline version may change as we improve and modify the pipeline
INFERENCE_PIPELINE_VERSION = "v01" 

# DATACLASSES TO DEFINE LOG ENTRY
@dataclass
class Spans:
    start: int
    end: int
    text: str
    label: str  #Enum['O', 'SYMPTOM_POS', 'SYMPTOM_NEG', f'CONFLICT-{some variable}']
@dataclass
class Reviewer(Enum):
    HUMAN = "HUMAN"
    AI = "AI"
    CODE = "CODE"
@dataclass
class LogEntry:
    date: str
    reviewer: Reviewer  # Only Reviewer.HUMAN or Reviewer.AI
    input_text: str
    tokens: List[str]
    token_level_labels: List[str]
    word_level_labels: List[str]  
    predicted_spans: List[Spans]
    expected_entities: List[Spans]
    error_type: Optional[ErrorTaxonomy]  
    reasoning: str 
    model: str = MODEL_NAME
    model_version: str = VERSION
    inference_pipeline_version: str = INFERENCE_PIPELINE_VERSION

## **Load Required Data Files**

In [29]:
## Load Required Data Files

# Load id2label mapping
print("📂 Loading id2label mapping...")
with open("data/id2label.json", "r") as f:
    id2label = json.load(f)

# Convert id2label keys from strings to integers (JSON loads keys as strings)
if any(isinstance(k, str) for k in id2label.keys()):
    id2label = {int(k): v for k, v in id2label.items()}

print(f"✅ Loaded {len(id2label)} label mappings")
print(f"Labels: {id2label}")

# Load behavioural evaluation set
print("\n📂 Loading behavioural evaluation set...")
with open("behavioural_set.json", "r") as f:
    behavioural_set = json.load(f)

print(f"✅ Loaded {len(behavioural_set)} categories")
print(f"Categories: {list(behavioural_set.keys())}")

# Count total examples
total_examples = sum(len(examples) for examples in behavioural_set.values())
print(f"Total test examples: {total_examples}")

📂 Loading id2label mapping...
✅ Loaded 5 label mappings
Labels: {0: 'B-SYMPTOM_NEG', 1: 'B-SYMPTOM_POS', 2: 'I-SYMPTOM_NEG', 3: 'I-SYMPTOM_POS', 4: 'O'}

📂 Loading behavioural evaluation set...
✅ Loaded 7 categories
Categories: ['Core symptom mention (clean baseline)', 'Temporal + progression (very important clinically)', 'Negation & uncertainty (classic failure mode)', 'Multiple symptoms in one sentence (boundary stress test)', 'Long, realistic clinical sentences (THIS IS GOLD 🥇)', 'Attribution & causal language (often tricky)', '“Should NOT extract” edge cases (behavioral guardrails)']
Total test examples: 35


## **Load model from local directory or GCS**

In [30]:
# Load the model
GCS_MODEL_PATH = f"{VERSION}/runs/{MODEL_NAME}/run_{RUN_IDX}"
BUCKET_NAME = settings.BUCKET_NAME  # "ner_training_data_results"

# Create a local directory path for the model
LOCAL_MODEL_DIR = f"./downloaded_models/{MODEL_NAME}/run_{RUN_IDX}"

# Check if model already exists locally
if os.path.exists(LOCAL_MODEL_DIR) and os.path.isfile(os.path.join(LOCAL_MODEL_DIR, "config.json")):
    print(f"✅ Model found locally at {LOCAL_MODEL_DIR}")
    print("Skipping download from GCS.")
else:
    # Download the model directory from GCS if not found locally
    print(f"📥 Model not found locally. Downloading from gs://{BUCKET_NAME}/{GCS_MODEL_PATH}...")
    downloaded_path = download_from_gcs(
        gcs_path=GCS_MODEL_PATH,
        local_path=LOCAL_MODEL_DIR,
        bucket_name=BUCKET_NAME
    )
    if downloaded_path:
        print(f"✅ Download complete. Model saved to {LOCAL_MODEL_DIR}")

# Load the model
print(f"📂 Loading model from {LOCAL_MODEL_DIR}...")
model = AutoModelForTokenClassification.from_pretrained(LOCAL_MODEL_DIR)
tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_DIR)

# Move to device
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
    
print(f"Using device: {device}")
model.to(device)
model.eval()
print("✅ Model is ready to be used")

✅ Model found locally at ./downloaded_models/dmis-lab/biobert-base-cased-v1.1/run_2
Skipping download from GCS.
📂 Loading model from ./downloaded_models/dmis-lab/biobert-base-cased-v1.1/run_2...
Using device: mps
✅ Model is ready to be used


## **Build AI Evaluation Loop**

In [ ]:
from xai_sdk import Client
from xai_sdk.chat import system, user

from pydantic import BaseModel

class DummyInfo(BaseModel):
    message: str

client = Client(api_key=os.getenv("XAI_API_KEY"))
chat = client.chat.create(model="grok-4-1-fast-non-reasoning-latest")


# test_prompt = "Say hello in a JSON object under the key 'message'."
# chat.append(user(test_prompt))
# response, invoice = chat.parse(DummyInfo)

In [31]:
# Extract error type keys from ErrorTaxonomy dataclass
ErrorTypeKeys =ErrorTaxonomy.__dataclass_fields__.keys()
# Create a set of valid error type keys for validation
_valid_error_types = set(ErrorTypeKeys)

# Create an Enum from the valid error types
ErrorType = Enum('ErrorType', {k: k for k in _valid_error_types})

class LLMResponse(BaseModel):
    """
    Response from LLM categorizing errors in NER predictions.
    """
    error_type: Optional[ErrorType] = Field(
        default=None,
        description=f"The error category code. Must be one of: {_valid_error_types}"
    )
    reasoning: str = Field(
        description="Explanation of the error categorization, including which spans/entities are problematic and why"
    )
    

In [ ]:
SYSTEM_PROMPT = """You are an expert at analyzing Named Entity Recognition (NER) model predictions for clinical symptom extraction.
Your task is to categorize errors in model predictions by comparing predicted spans against expected entities.
SPAN Entities: SYMPTOM_POS (a symptom reported by the patient.), SYMPTOM_NEG (a symptom negated by the patient.), O, not an entity

## Error Taxonomy
You must classify errors into ONE of these 8 categories (or return None if no error):

IMPORTANT: Return the SHORT CODE (type_0, type_1, etc.) NOT the full name!

- type_0: Irrelevant Span Mislabeling
  - A non-relevant span is incorrectly detected as an entity
  - Example: `65` labeled as SYMPTOM_POS
  - Indicates semantic confusion—model cannot distinguish between entity and non-entity tokens

- type_1: Missing Expected Entity
  - A clinically relevant symptom is not detected at all
  - Example: `exanthema` not extracted
  - Strong signal for dataset enrichment

- type_2: Tokenization Artifacts
  - Errors caused by subword splits influencing predictions
  - Example: Mixed labels across subwords of a single word, conflicting labels aggregated into `CONFLICT-*`
  - Expected behavior with WordPiece/BPE tokenizers

- type_3: Incorrect Polarity Assignment
  - Incorrect polarity assignment in the presence of negation
  - Example: `muscle cramps` labeled as `SYMPTOM_NEG` when context implies presence
  - Model struggles with negation scope and contrastive clauses

- type_4: BIO Sequencing Errors
  - Incorrect or inconsistent BIO tag transitions
  - Example: `I-SYMPTOM_POS` without a preceding `B-`, new symptom starting with `I-` instead of `B-`
  - Model uncertainty at entity boundaries

- type_5: Boundary Overreach
  - Model captures a symptom PLUS unrelated surrounding words
  - Example: `blisters on his head` (should be just `blisters`)
  - Core entity detected correctly but span precision is low

- type_6: Boundary Undereach
  - Model captures only part of a multi-word symptom
  - Example: `chest` `pain` instead of `chest pain`
  - Partial entity detection

- type_7: Model Overgeneralization
  - Model predicts a symptom where none exists
  - Example: Predicting `SYMPTOM_POS` for person names (`Marie`), general states (`well`), demographics (`65`)
  - Model has learned overly broad symptom cues

## Input Format

You will receive:
- `input_text`: Original text
- `tokens`: Tokenized tokens (filtered, no special tokens)
- `token_level_labels`: BIO labels for each token (e.g., "B-SYMPTOM_POS", "I-SYMPTOM_POS", "O")
- `word_ids`: Word index for each token (tokens from same word share same word_id)
- `words`: Naive word split of text
- `word_level_labels`: Aggregated BIO labels per word
- `predicted_spans`: List of predicted entity spans with {start, end, text, label}
- `expected_entities`: List of expected entity text strings (from ground truth)


## Your Task

1. Compare predicted_spans against expected_entities
2. Identify discrepancies (missing entities, extra entities, wrong boundaries, wrong polarity)
3. Classify the PRIMARY error type (choose the most significant issue)
4. Return the SHORT CODE (type_0, type_1, type_2, etc.) in the error_type field
5. Provide clear reasoning explaining:
   - Which spans/entities are problematic
   - Why this error type was chosen
   - What the model got right vs wrong
   - No more than 30 words per explanation. Be concise.


- If prediction matches expected perfectly, return error_type=None
- If multiple error types apply, note all errors
- CRITICAL: Use the exact short code format (type_0, type_1, etc.) - do NOT use full names
- Be specific in reasoning—reference actual spans and text
- Consider tokenization artifacts when spans don't align perfectly
- Pay attention to BIO tag sequences and polarity (POS vs NEG)"""

In [32]:
behavioural_set

{'Core symptom mention (clean baseline)': [{'example': 'Patient reports mouth symptom.',
   'entities': ['mouth symptom']},
  {'example': 'The main complaint today is clonic seizure.',
   'entities': ['clonic seizure']},
  {'example': 'Complains of bradypnea since yesterday.',
   'entities': ['bradypnea']},
  {'example': 'Experiencing dysphonia intermittently.',
   'entities': ['dysphonia']},
  {'example': 'Presents with joint inflammation.',
   'entities': ['joint inflammation']}],
 'Temporal + progression (very important clinically)': [{'example': 'Patient reports localized superficial lump that started two days ago.',
   'entities': ['localized superficial lump']},
  {'example': 'Complains of corkscrew hair, which has been worsening over the past week.',
   'entities': ['corkscrew hair']},
  {'example': 'Describes right lower quadrant abdominal pain that began suddenly this morning.',
   'entities': ['right lower quadrant abdominal pain']},
  {'example': 'Has had urinary stream symp

In [33]:
text = behavioural_set['Core symptom mention (clean baseline)'][0]['example']
print(f"TEXT: {text}")
tokens, token_labels, word_ids, words, word_labels = predict_word_level(
        text=text,
        model=model,
        tokenizer=tokenizer,
        id2label=id2label,
        device=device,
    )
spans = word_labels_to_spans(words, word_labels)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


TEXT: Patient reports mouth symptom.


# Error Categorization Loop